In [1]:
import os
import json
import shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from skimage.io import imread, imsave
from tqdm import tqdm
import albumentations as A
from ultralytics import YOLO

# Paths
DATA_DIR = os.path.expanduser("~/cp-anemia-detection/data/fingernail-anemia")
CSV_PATH = os.path.join(DATA_DIR, "metadata.csv")
metadata = pd.read_csv(CSV_PATH)
YOLO_DATASET_DIR = os.path.join(os.path.expanduser("~/cp-anemia-detection/data"), "fingernail-anemia-yolo")
IMG_DIR = os.path.join(YOLO_DATASET_DIR, "images")
LBL_DIR = os.path.join(YOLO_DATASET_DIR, "labels")

/home/sebastian-cruz6/cp-anemia-detection/cawt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
metadata["HB_LEVEL_GperDeciL"] = metadata["HB_LEVEL_GperL"]/10
metadata['NAIL_BOUNDING_BOXES'] = metadata['NAIL_BOUNDING_BOXES'].apply(json.loads)
metadata['SKIN_BOUNDING_BOXES'] = metadata['SKIN_BOUNDING_BOXES'].apply(json.loads)

# Create directory structure
for split in ["train", "val"]:
    os.makedirs(os.path.join(IMG_DIR, split), exist_ok=True)
    os.makedirs(os.path.join(LBL_DIR, split), exist_ok=True)

# Split data
train_df, val_df = train_test_split(metadata, test_size=0.2, random_state=42)

# Define augmentations
train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Rotate(limit=10, p=0.5)
], bbox_params=A.BboxParams(format='yolo', label_fields=['category_ids']))

# Number of augmented copies per image
N_AUGS = 3

def convert_and_save(df, split, augment=False):
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split} set"):

        label = 'Anemic' if row.HB_LEVEL_GperDeciL < 12.0 else 'Non-anemic'
        image_path = os.path.join(DATA_DIR ,label, f'{row.PATIENT_ID}.jpg')

        if not os.path.isfile(image_path):
            print(f"Image not found: {image_path}")
            continue

        image = imread(image_path)
        height, width = image.shape[:2]

        bboxes = []
        for bbox in row["NAIL_BOUNDING_BOXES"]:
            if not isinstance(bbox, list) or len(bbox) != 4:
                continue
            top, left, bottom, right = bbox
            x_center = ((left + right) / 2) / width
            y_center = ((top + bottom) / 2) / height
            bbox_width = (right - left) / width
            bbox_height = (bottom - top) / height
            bboxes.append([x_center, y_center, bbox_width, bbox_height])

        category_ids = [0] * len(bboxes)  # Assuming only one class \"nail\"

        if augment and split == "train":
            for aug_idx in range(N_AUGS):
                augmented = train_transforms(image=image, bboxes=bboxes, category_ids=category_ids)
                aug_image = augmented['image']
                aug_bboxes = augmented['bboxes']

                # Save augmented image
                aug_image_name = f"{row.PATIENT_ID}_aug{aug_idx}.jpg"
                aug_label_name = f"{row.PATIENT_ID}_aug{aug_idx}.txt"

                aug_img_path = os.path.join(IMG_DIR, split, aug_image_name)
                aug_lbl_path = os.path.join(LBL_DIR, split, aug_label_name)

                imsave(aug_img_path, aug_image)

                with open(aug_lbl_path, 'w') as f:
                    for bbox in aug_bboxes:
                        x_center, y_center, bbox_width, bbox_height = bbox
                        f.write(f"0 {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}\n")

        # Always save the original image
        orig_img_path = os.path.join(IMG_DIR, split, f"{row.PATIENT_ID}.jpg")
        orig_lbl_path = os.path.join(LBL_DIR, split, f"{row.PATIENT_ID}.txt")

        shutil.copy(image_path, orig_img_path)

        with open(orig_lbl_path, 'w') as f:
            for bbox in bboxes:
                x_center, y_center, bbox_width, bbox_height = bbox
                f.write(f"0 {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}\n")

# Rerun with augmentation for training set
convert_and_save(train_df, "train", augment=True)
convert_and_save(val_df, "val", augment=False)

Processing val set: 100%|██████████| 50/50 [00:00<00:00, 166.44it/s]


In [3]:
data_yaml = """
path: {}
train: images/train
val: images/val
names:
  0: fingernail
""".format(YOLO_DATASET_DIR)

with open(os.path.join(YOLO_DATASET_DIR, "fingernail.yaml"), "w") as f:
    f.write(data_yaml)

In [4]:
# from ultralytics import YOLO

# # Load pretrained YOLOv8n model
# model = YOLO("yolov8n.pt")

# # Train
# model.train(
#     data=os.path.join(YOLO_DATASET_DIR, "fingernail.yaml"),
#     epochs=100,
#     imgsz=640,
#     batch=16,
#     project="yolo_fingernail_detection",
#     name="yolov8n_fingernail",
#     exist_ok=True
# )

# model = YOLO("model_compression/yolov8n_fingernail/weights/best.pt")
# results = model.predict(source=os.path.join(YOLO_DATASET_DIR, "images/val"), save=True)

In [5]:
# # Load pretrained YOLOv8n model
model = YOLO("yolov8n.pt")

resource_opt_config = {
    "epochs": 50,                   # Start small; tune based on training curve
    "imgsz": 640,                   # Smaller image = faster and smaller model
    "batch": 16,                    # Keep this low to prevent memory overload
    "lr0": 0.003,                   # Lower LR to prevent overshooting
    "weight_decay": 0.0002,         # Lower = less regularization
    "momentum": 0.9,
    "warmup_epochs": 3.0,
    "optimizer": "SGD",             # Lightweight vs. AdamW
    "hsv_h": 0.01,                  # Reduce augmentations — more stable for small models
    "hsv_s": 0.3,
    "hsv_v": 0.3,
    "scale": 0.3,
    "translate": 0.0,               # Disable unnecessary augmentations
    "fliplr": 0.5,                  # Horizontal flip only
    "mosaic": 0.0,                  # Turn off mosaic (not ideal for small datasets)
    "mixup": 0.0,                   # Disable mixup
    "project": "fingernail_resource_opt",
    "name": "yolov8n_optimized",
    "exist_ok": True
}

model.train(
    data=os.path.join(YOLO_DATASET_DIR, "fingernail.yaml"),
    **resource_opt_config
)

# Load best model
model = YOLO("fingernail_resource_opt/yolov8n_optimized/weights/best.pt")

# Export to ONNX, TorchScript, or TFLite
# model.export(format="onnx")       # For edge AI engines
# model.export(format="torchscript") # For PyTorch-based mobile
# model.export(format="tflite")      # For Android / TF Lite runtimes

New https://pypi.org/project/ultralytics/8.3.116 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.115 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/home/sebastian-cruz6/cp-anemia-detection/data/fingernail-anemia-yolo/fingernail.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=fingernail_resource_opt, name=yolov8n_optimized, exist_ok=True, pretrained=True, optimizer=SGD, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnos

train: Scanning /home/sebastian-cruz6/cp-anemia-detection/data/fingernail-anemia-yolo/labels/train... 800 images, 0 backgrounds, 0 corrupt: 100%|██████████| 800/800 [00:00<00:00, 2215.98it/s]

train: New cache created: /home/sebastian-cruz6/cp-anemia-detection/data/fingernail-anemia-yolo/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5557.2±3587.3 MB/s, size: 371.3 KB)


val: Scanning /home/sebastian-cruz6/cp-anemia-detection/data/fingernail-anemia-yolo/labels/val.cache... 50 images, 0 backgrounds, 0 corrupt: 100%|██████████| 50/50 [00:00<?, ?it/s]


...

      50/50      2.23G     0.6031     0.3521     0.8721         48        640: 100%|██████████| 50/50 [00:01<00:00, 28.86it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:00<00:00, 24.24it/s]

                   all         50        150      0.993          1      0.995      0.673



50 epochs completed in 0.027 hours.
Optimizer stripped from fingernail_resource_opt/yolov8n_optimized/weights/last.pt, 6.2MB
Optimizer stripped from fingernail_resource_opt/yolov8n_optimized/weights/best.pt, 6.2MB

Validating fingernail_resource_opt/yolov8n_optimized/weights/best.pt...
Ultralytics 8.3.115 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:00<00:00, 19.62it/s]


                   all         50        150      0.993          1      0.995      0.692
Speed: 0.1ms preprocess, 0.2ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to fingernail_resource_opt/yolov8n_optimized


In [34]:
import os
from ultralytics import YOLO

# Step 1: Save YAML config
data_yaml = """
path: {}
train: images/train
val: images/val
names:
  0: fingernail
""".format(YOLO_DATASET_DIR)

yaml_path = os.path.join(YOLO_DATASET_DIR, "fingernail.yaml")
with open(yaml_path, "w") as f:
    f.write(data_yaml)

# Step 2: First Training (optional)
# model = YOLO("yolov8n.pt")
# model.train(
#     data=yaml_path,
#     epochs=100,
#     imgsz=640,
#     batch=16,
#     project="yolo_fingernail_detection",
#     name="yolov8n_fingernail",
#     exist_ok=True
# )

# Step 3: Resource-Optimized Training
# Reload a fresh model
model = YOLO("yolov8n.pt")

resource_opt_config = {
    "epochs": 50,
    "imgsz": 224,
    "batch": 16,
    "lr0": 0.003,
    "weight_decay": 0.0002,
    "momentum": 0.9,
    "warmup_epochs": 3.0,
    "optimizer": "SGD",
    "hsv_h": 0.01,
    "hsv_s": 0.3,
    "hsv_v": 0.3,
    "scale": 0.3,
    "translate": 0.0,
    "fliplr": 0.5,
    "mosaic": 0.0,
    "mixup": 0.0,
    "project": "fingernail_resource_opt",
    "name": "yolov8n_optimized",
    "exist_ok": True,
    "int8":True
}

model.train(
    data=yaml_path,
    **resource_opt_config
)

# Step 4: Load best model
model = YOLO("fingernail_resource_opt/yolov8n_optimized/weights/best.pt")

New https://pypi.org/project/ultralytics/8.3.116 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.115 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/home/sebastian-cruz6/cp-anemia-detection/data/fingernail-anemia-yolo/fingernail.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=224, save=True, save_period=-1, cache=False, device=None, workers=8, project=fingernail_resource_opt, name=yolov8n_optimized, exist_ok=True, pretrained=True, optimizer=SGD, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnos

train: Scanning /home/sebastian-cruz6/cp-anemia-detection/data/fingernail-anemia-yolo/labels/train.cache... 800 images, 0 backgrounds, 0 corrupt: 100%|██████████| 800/800 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2507.8±2528.9 MB/s, size: 371.3 KB)


val: Scanning /home/sebastian-cruz6/cp-anemia-detection/data/fingernail-anemia-yolo/labels/val.cache... 50 images, 0 backgrounds, 0 corrupt: 100%|██████████| 50/50 [00:00<?, ?it/s]


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      50/50     0.424G     0.8587     0.4757     0.8006         48        224: 100%|██████████| 50/50 [00:01<00:00, 38.48it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:00<00:00, 27.99it/s]


                   all         50        150      0.999          1      0.995      0.611

50 epochs completed in 0.020 hours.
Optimizer stripped from fingernail_resource_opt/yolov8n_optimized/weights/last.pt, 6.2MB
Optimizer stripped from fingernail_resource_opt/yolov8n_optimized/weights/best.pt, 6.2MB

Validating fingernail_resource_opt/yolov8n_optimized/weights/best.pt...
Ultralytics 8.3.115 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:00<00:00, 13.50it/s]


                   all         50        150      0.999          1      0.995      0.638
Speed: 0.0ms preprocess, 0.1ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to fingernail_resource_opt/yolov8n_optimized


In [35]:
import torch

weights_path = "/home/sebastian-cruz6/cp-anemia-detection/notebooks/model_compression/fingernail_resource_opt/yolov8n_optimized/weights"

# 1. Export correctly
model.export(format='torchscript', simplify=True, optimize=True)

# 2. Load and validate native TorchScript model
scripted_model = torch.jit.load(os.path.join(weights_path, "best.torchscript"))
# Validate it with inputs normalized between 0 and 1!

# 3. Quantize dynamically
quantized_model = torch.quantization.quantize_dynamic(
    scripted_model,
    {torch.nn.Linear},  # Quantize Linear layers mainly
    dtype=torch.qint8
)
# Save quantized model
quantized_model.save(os.path.join(weights_path, "best_quantized.torchscript.pt"))

Ultralytics 8.3.115 🚀 Python-3.12.3 torch-2.6.0+cu124 CPU (AMD Ryzen Threadripper 7960X 24-Cores)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'fingernail_resource_opt/yolov8n_optimized/weights/best.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 5, 1029) (6.0 MB)

TorchScript: starting export with torch 2.6.0+cu124...
TorchScript: optimizing for mobile...
TorchScript: export success ✅ 0.7s, saved as 'fingernail_resource_opt/yolov8n_optimized/weights/best.torchscript' (11.5 MB)

Export complete (0.8s)
Results saved to /home/sebastian-cruz6/cp-anemia-detection/notebooks/model_compression/fingernail_resource_opt/yolov8n_optimized/weights
Predict:         yolo predict task=detect model=fingernail_resource_opt/yolov8n_optimized/weights/best.torchscript imgsz=224  
Validate:        yolo val task=detect model=fingernail_resource_opt/yolov8n_optimized/weights/best.torchscript imgsz=224 data=/home/sebastian-cruz6/cp-

In [38]:
import torch
import time
from ultralytics import YOLO

# Load original YOLOv8 .pt
model_native = YOLO("fingernail_resource_opt/yolov8n_optimized/weights/best.pt")

# Dummy input (batch size 1, 3 channels, 416x416 resolution)
dummy_input = torch.randn(1, 3, 224, 224).to('cpu')

# Measure inference time (native YOLOv8)
start_time = time.time()
results = model_native.predict(source=dummy_input, imgsz=640, device='cpu', verbose=False)
end_time = time.time()

print(f"✅ Native YOLOv8 model inference time: {(end_time - start_time)*1000:.2f} ms")

WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.713801383972168. Dividing input by 255.
✅ Native YOLOv8 model inference time: 46.96 ms


In [39]:
# Load standard TorchScript model
torchscript_path = "/home/sebastian-cruz6/cp-anemia-detection/notebooks/model_compression/fingernail_resource_opt/yolov8n_optimized/weights/best.torchscript"
model_scripted = torch.jit.load(torchscript_path)
model_scripted.eval()

# Measure TorchScript inference
start_time = time.time()
output = model_scripted(dummy_input)
end_time = time.time()

print(f"✅ TorchScript model inference time: {(end_time - start_time)*1000:.2f} ms")

✅ TorchScript model inference time: 18.95 ms


In [40]:
# Dynamic quantization (works best for CPU inference)
model_quantized = torch.quantization.quantize_dynamic(
    model_scripted, {torch.nn.Linear}, dtype=torch.qint8
)

# Save quantized model
torch.jit.save(model_quantized, "fingernail_resource_opt/yolov8n_optimized/weights/best_quantized.torchscript.pt")
print("✅ Quantized model saved.")

✅ Quantized model saved.


In [41]:
# Load quantized model
torchscript_quantized_path = "/home/sebastian-cruz6/cp-anemia-detection/notebooks/model_compression/fingernail_resource_opt/yolov8n_optimized/weights/best_quantized.torchscript.pt"
model_quantized_loaded = torch.jit.load(torchscript_quantized_path)
model_quantized_loaded.eval()

# Measure Quantized TorchScript inference
start_time = time.time()
output = model_quantized_loaded(dummy_input)
end_time = time.time()

print(f"✅ Quantized TorchScript model inference time: {(end_time - start_time)*1000:.2f} ms")


✅ Quantized TorchScript model inference time: 21.50 ms


In [44]:
import os
import glob
import torch
import numpy as np
import cv2
from tqdm import tqdm
from ultralytics import YOLO

# Helper: Basic Non-Maximum Suppression
def non_max_suppression(predictions, conf_thres=0.25, iou_thres=0.45):
    boxes = predictions[predictions[:, 4] > conf_thres]
    if boxes.shape[0] == 0:
        return []

    scores = boxes[:, 4]
    indices = torch.argsort(scores, descending=True)
    boxes = boxes[indices]

    keep_boxes = []
    while boxes.size(0):
        box = boxes[0]
        keep_boxes.append(box)
        if boxes.size(0) == 1:
            break
        ious = compute_iou_torch(box[:4].unsqueeze(0), boxes[1:, :4])
        boxes = boxes[1:][ious < iou_thres]

    return torch.stack(keep_boxes) if keep_boxes else torch.empty((0, 6))

def compute_iou_torch(box1, boxes2):
    x1 = torch.max(box1[:, 0], boxes2[:, 0])
    y1 = torch.max(box1[:, 1], boxes2[:, 1])
    x2 = torch.min(box1[:, 2], boxes2[:, 2])
    y2 = torch.min(box1[:, 3], boxes2[:, 3])

    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1 + area2 - inter

    return inter / (union + 1e-6)

# Helper: IoU for numpy arrays
def compute_iou_np(box1, box2):
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / (union_area + 1e-6)

# Helper: Load labels
def load_ground_truth(label_path, img_width, img_height):
    gt_boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, x_center, y_center, width, height = map(float, parts)
            x1 = (x_center - width / 2) * img_width
            y1 = (y_center - height / 2) * img_height
            x2 = (x_center + width / 2) * img_width
            y2 = (y_center + height / 2) * img_height
            gt_boxes.append([x1, y1, x2, y2])
    return gt_boxes

# Evaluation function
def evaluate_model(model_path, images_dir, labels_dir, model_type="torchscript", imgsz=224, conf_thresh=0.25):
    print(f"\nEvaluating model: {model_path} (type: {model_type})")

    if model_type == "native":
        model = YOLO(model_path)
    else:
        model = torch.jit.load(model_path)
        model.eval()

    image_paths = glob.glob(os.path.join(images_dir, "*.jpg"))

    all_ious = []
    TP, FP, FN = 0, 0, 0

    for img_path in tqdm(image_paths, desc="Evaluating"):
        filename = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(labels_dir, f"{filename}.txt")

        if not os.path.exists(label_path):
            continue

        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, (imgsz, imgsz))
        img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).unsqueeze(0).float() / 255.0

        if model_type == "native":
            preds = model.predict(img_tensor, imgsz=imgsz, verbose=False)[0].boxes
            pred_boxes = preds.xyxy.cpu().numpy() if preds is not None else []
        else:
            with torch.no_grad():
                preds = model(img_tensor)[0]
            preds = non_max_suppression(preds, conf_thresh)
            pred_boxes = preds[:, :4].cpu().numpy() if preds.shape[0] else []

        gt_boxes = load_ground_truth(label_path, imgsz, imgsz)

        matched = set()
        for pred_box in pred_boxes:
            best_iou = 0
            best_gt_idx = -1
            for idx, gt_box in enumerate(gt_boxes):
                iou = compute_iou_np(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx

            if best_iou >= 0.5:
                TP += 1
                matched.add(best_gt_idx)
                all_ious.append(best_iou)
            else:
                FP += 1

        FN += len(gt_boxes) - len(matched)

    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)
    mAP50 = np.mean(all_ious) if all_ious else 0.0

    print(f"✅ Precision: {precision:.4f}")
    print(f"✅ Recall: {recall:.4f}")
    print(f"✅ mAP@0.5: {mAP50:.4f}")
    print(f"✅ Total Images Evaluated: {len(image_paths)}")
    print(f"✅ TP: {TP}, FP: {FP}, FN: {FN}")

    return precision, recall, mAP50

In [45]:
# Paths
images_val_dir = os.path.join(YOLO_DATASET_DIR, "images", "val")
labels_val_dir = os.path.join(YOLO_DATASET_DIR, "labels", "val")

model_native = "fingernail_resource_opt/yolov8n_optimized/weights/best.pt"
torchscript_quantized_path = "/home/sebastian-cruz6/cp-anemia-detection/notebooks/model_compression/fingernail_resource_opt/yolov8n_optimized/weights/best_quantized.torchscript.pt"
torchscript_path = "/home/sebastian-cruz6/cp-anemia-detection/notebooks/model_compression/fingernail_resource_opt/yolov8n_optimized/weights/best.torchscript"

evaluate_model(model_native, images_val_dir, labels_val_dir, model_type="native")

# Evaluate Standard TorchScript Model
evaluate_model(torchscript_path, images_val_dir, labels_val_dir, model_type="torchscript")

# Evaluate Quantized TorchScript Model
evaluate_model(torchscript_quantized_path, images_val_dir, labels_val_dir, model_type="torchscript")


Evaluating model: fingernail_resource_opt/yolov8n_optimized/weights/best.pt (type: native)


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 50/50 [00:00<00:00, 101.93it/s]


✅ Precision: 0.9740
✅ Recall: 1.0000
✅ mAP@0.5: 0.8068
✅ Total Images Evaluated: 50
✅ TP: 150, FP: 4, FN: 0

Evaluating model: /home/sebastian-cruz6/cp-anemia-detection/notebooks/model_compression/fingernail_resource_opt/yolov8n_optimized/weights/best.torchscript (type: torchscript)


Evaluating: 100%|██████████| 50/50 [00:00<00:00, 55.51it/s]


✅ Precision: 0.0000
✅ Recall: 0.0000
✅ mAP@0.5: 0.0000
✅ Total Images Evaluated: 50
✅ TP: 0, FP: 200, FN: 150

Evaluating model: /home/sebastian-cruz6/cp-anemia-detection/notebooks/model_compression/fingernail_resource_opt/yolov8n_optimized/weights/best_quantized.torchscript.pt (type: torchscript)


Evaluating: 100%|██████████| 50/50 [00:00<00:00, 55.45it/s]

✅ Precision: 0.0000
✅ Recall: 0.0000
✅ mAP@0.5: 0.0000
✅ Total Images Evaluated: 50
✅ TP: 0, FP: 200, FN: 150


(0.0, 0.0, 0.0)